# Sistema de Alerta Temprana (EWS) de Estrés Financiero Sistémico

**TFM — Máster en Ciencia de Datos**

Este notebook contiene el código fuente completo del trabajo. El informe de 20 páginas orientado a negocio se entrega como documento Word aparte; aquí se documentan las decisiones técnicas con el detalle necesario para su reproducibilidad.

**Estructura de este notebook**:

1. Configuración del entorno
2. Ingesta de datos desde la API de FRED
3. Remuestreo a frecuencia semanal
4. Análisis exploratorio de datos (EDA)
5. Feature engineering
6. Modelización: Ridge, LightGBM (walk-forward), LSTM
7. Backtesting sobre GFC 2008, COVID 2020 y SVB 2023
8. Interpretabilidad (SHAP)
9. Productivización: exportación de artefactos para el dashboard (`streamlit_app/`, fuera de este notebook)

## 0. Configuración del entorno

In [ ]:
# Librerías no incluidas por defecto en el runtime de Colab
!pip install -q fredapi lightgbm shap

In [ ]:
import os
import warnings
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fredapi import Fred

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

### Clave de API de FRED

La API de FRED es pública y gratuita. Para obtener una clave:

1. Crear una cuenta en https://fred.stlouisfed.org/
2. Solicitar la API key en https://fred.stlouisfed.org/docs/api/api_key.html
3. En Colab, guardarla como *secret* (icono 🔑 en el panel lateral izquierdo) con el nombre `FRED_API_KEY`, y dar acceso al notebook.

Si no se encuentra como secret, el notebook la pedirá por teclado (no queda guardada en el código ni en el propio notebook).

In [ ]:
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get('FRED_API_KEY')
except Exception:
    FRED_API_KEY = None

if not FRED_API_KEY:
    from getpass import getpass
    FRED_API_KEY = getpass('Introduce tu API key de FRED: ')

fred = Fred(api_key=FRED_API_KEY)
print('Conexión a FRED configurada.')

## 1. Configuración de series y parámetros

El dataset se construye íntegramente a partir de series públicas de FRED, todas con histórico desde antes de 2000. `NFCI` es la variable objetivo (target); el resto son predictores.

| Serie | Descripción | Frecuencia nativa | Rol |
|---|---|---|---|
| NFCI | National Financial Conditions Index (Fed de Chicago) | Semanal | Target |
| BAA10Y | Spread corporativo Baa (Moody's) sobre Treasury 10Y | Diaria | Estrés crédito |
| VIXCLS | CBOE Volatility Index | Diaria | Volatilidad implícita |
| T10Y2Y | Spread curva de tipos 10Y-2Y | Diaria | Estructura temporal |
| TEDRATE | TED spread | Diaria (discontinuada en 2022) | Estrés interbancario |
| DGS10 | Treasury 10 años | Diaria | Tipo libre de riesgo |
| UNRATE | Tasa de desempleo | Mensual | Actividad real |
| CPIAUCSL | IPC (se deriva inflación interanual) | Mensual | Actividad real |
| DCOILWTICO | Precio del petróleo WTI | Diaria | Materias primas |

**Nota sobre `TEDRATE`**: la Fed discontinuó esta serie el 3 de enero de 2022 tras el cese de publicación del LIBOR a 3 meses. Se mantiene como predictor histórico (relevante en la GFC 2008), pero a partir de esa fecha queda como `NaN` en lugar de arrastrar el último valor indefinidamente (ver parámetro `max_ffill_days` más abajo). Es una limitación conocida del dataset, documentada en el informe.

In [ ]:
START_DATE = '2000-01-01'
END_DATE = date.today().isoformat()

# max_ffill_days: nº máximo de días que se arrastra hacia delante (ffill) un valor antes de
# considerarlo obsoleto y convertirlo en NaN. Evita que una serie discontinuada (p.ej. TEDRATE)
# quede "congelada" con su último valor real durante años.
SERIES = {
    'NFCI':       {'name': 'nfci',       'freq': 'W', 'max_ffill_days': 10, 'role': 'target'},
    'BAA10Y':     {'name': 'baa10y',     'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
    'VIXCLS':     {'name': 'vix',        'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
    'T10Y2Y':     {'name': 't10y2y',     'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
    'TEDRATE':    {'name': 'ted_spread', 'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
    'DGS10':      {'name': 'dgs10',      'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
    'UNRATE':     {'name': 'unrate',     'freq': 'M', 'max_ffill_days': 45, 'role': 'feature'},
    'CPIAUCSL':   {'name': 'cpi',        'freq': 'M', 'max_ffill_days': 45, 'role': 'feature'},
    'DCOILWTICO': {'name': 'wti',        'freq': 'D', 'max_ffill_days': 10, 'role': 'feature'},
}

# Episodios de estrés histórico utilizados en EDA y, más adelante, en el backtesting
CRISIS_PERIODS = {
    'GFC 2008':   ('2007-12-01', '2009-06-30'),
    'COVID 2020': ('2020-02-15', '2020-06-30'),
    'SVB 2023':   ('2023-03-01', '2023-05-31'),
}

print(f'Periodo de análisis: {START_DATE} → {END_DATE}')
print(f'Nº de series: {len(SERIES)}')

## 2. Ingesta de datos desde FRED

Se descarga cada serie a su frecuencia nativa (sin transformar) y se cachea localmente en CSV para no repetir llamadas a la API en ejecuciones sucesivas del notebook.

In [ ]:
RAW_DATA_DIR = 'data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)


def fetch_fred_series(series_id: str, use_cache: bool = True) -> pd.Series:
    """Descarga una serie de FRED (o la lee de caché local si ya existe)."""
    cache_path = os.path.join(RAW_DATA_DIR, f'{series_id}.csv')

    if use_cache and os.path.exists(cache_path):
        s = pd.read_csv(cache_path, index_col=0, parse_dates=True).iloc[:, 0]
        return s

    s = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
    s.index.name = 'date'
    s.to_csv(cache_path)
    return s


raw_series = {}
for series_id, meta in SERIES.items():
    s = fetch_fred_series(series_id)
    raw_series[series_id] = s
    print(f'{series_id:12s} {meta["freq"]:2s}  {s.index.min().date()} → {s.index.max().date()}  '
          f'({len(s)} obs, {s.isna().sum()} NaN)')

In [ ]:
# Vista rápida de las últimas observaciones de cada serie (a su frecuencia nativa)
raw_preview = pd.DataFrame({sid: s for sid, s in raw_series.items()})
raw_preview.tail()

## 3. Remuestreo a frecuencia semanal

`NFCI` se publica semanalmente con fecha de viernes. Para alinear el resto de series (diarias y mensuales) a esa misma rejilla temporal, cada serie se propaga hacia delante (`ffill`) a frecuencia diaria y después se muestrea el valor del viernes de cada semana — es decir, "el último dato disponible a cierre de esa semana", que es exactamente cómo se interpretaría la señal en un uso real. El límite `max_ffill_days` evita arrastrar valores obsoletos cuando una serie dejó de publicarse o tiene huecos anómalos.

In [ ]:
WEEKLY_INDEX = pd.date_range(START_DATE, END_DATE, freq='W-FRI')
ALL_DAYS = pd.date_range(START_DATE, END_DATE, freq='D')


def to_weekly(series: pd.Series, max_ffill_days: int) -> pd.Series:
    """Reindexa una serie de cualquier frecuencia a la rejilla semanal (viernes),
    propagando el último valor conocido con un límite de días para evitar datos obsoletos."""
    s = series.dropna().sort_index()
    daily = s.reindex(s.index.union(ALL_DAYS)).ffill(limit=max_ffill_days).reindex(ALL_DAYS)
    return daily.reindex(WEEKLY_INDEX)


weekly_cols = {}
for series_id, meta in SERIES.items():
    weekly_cols[meta['name']] = to_weekly(raw_series[series_id], meta['max_ffill_days'])

df_weekly = pd.DataFrame(weekly_cols, index=WEEKLY_INDEX)
df_weekly.index.name = 'date'

# La inflación interanual es más informativa que el nivel del IPC como señal de estrés/actividad
cpi_yoy = raw_series['CPIAUCSL'].pct_change(12) * 100
df_weekly['cpi_yoy'] = to_weekly(cpi_yoy, SERIES['CPIAUCSL']['max_ffill_days'])
df_weekly = df_weekly.drop(columns=['cpi'])

df_weekly = df_weekly.dropna(subset=['nfci'])  # el target debe estar siempre presente
print(df_weekly.shape)
df_weekly.tail()

## 4. Análisis exploratorio de datos (EDA)

In [ ]:
missing_pct = df_weekly.isna().mean().sort_values(ascending=False) * 100
print('% de valores ausentes por variable (rejilla semanal):')
missing_pct.round(2)

In [ ]:
df_weekly.describe().T

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(13, 7))

axes[0].plot(df_weekly.index, df_weekly['nfci'], color='#1f77b4', linewidth=1)
axes[0].axhline(0, color='grey', linewidth=0.8, linestyle='--')
axes[0].set_title('NFCI — National Financial Conditions Index')
axes[0].set_ylabel('Índice (0 = condiciones medias)')

axes[1].plot(df_weekly.index, df_weekly['baa10y'], color='#d62728', linewidth=1)
axes[1].set_title('BAA10Y — Spread corporativo Baa sobre Treasury 10Y')
axes[1].set_ylabel('Puntos porcentuales')

colors = ['#ff7f0e', '#2ca02c', '#9467bd']
for ax in axes:
    for (label, (start, end)), color in zip(CRISIS_PERIODS.items(), colors):
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=color, alpha=0.15)

axes[0].legend([plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.3) for c in colors],
               CRISIS_PERIODS.keys(), loc='upper left', ncol=3, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
corr = df_weekly.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'label': 'Correlación'})
plt.title('Correlación entre variables (niveles, frecuencia semanal)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
cols_to_plot = [c for c in df_weekly.columns if c != 'nfci']

for ax, col in zip(axes.flat, cols_to_plot):
    sns.histplot(df_weekly[col].dropna(), kde=True, ax=ax, color='#4c72b0')
    ax.set_title(col)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

### Estacionariedad

Se aplica el test aumentado de Dickey-Fuller (ADF) a cada serie para valorar si conviene modelizar en niveles o en diferencias. `NFCI`, al ser ya un índice estandarizado (media ≈ 0), suele comportarse como estacionario o cercano a ello; variables como `dgs10` o `wti` son típicamente no estacionarias en niveles. Esta información se retoma en la fase de modelización para decidir qué transformaciones aplicar a cada familia de modelos.

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_results = []
for col in df_weekly.columns:
    series = df_weekly[col].dropna()
    stat, pvalue, *_ = adfuller(series)
    adf_results.append({'variable': col, 'adf_stat': stat, 'p_value': pvalue,
                         'estacionaria (5%)': pvalue < 0.05})

pd.DataFrame(adf_results).set_index('variable').round(4)

## 5. Feature engineering

Se generan dos familias de variables predictoras, ambas construidas exclusivamente con información pasada respecto a cada fecha (sin usar el propio valor contemporáneo salvo donde se indique), para que sean utilizables en un escenario real de predicción:

- **Lags**: valor de cada variable hace 1, 2, 4, 8 y 12 semanas.
- **Estadísticos móviles**: media y desviación típica en ventanas de 4, 8 y 12 semanas, calculadas sobre la serie ya desplazada una semana (evita usar el dato de la propia semana de referencia).

La definición del horizonte de predicción (`h` semanas vista) y del umbral de estrés para la clasificación binaria se pospone a la fase de modelización, donde se calculará con una ventana expansiva (solo información pasada en cada punto) para evitar *look-ahead bias* — un umbral fijo calculado con todo el histórico filtraría información del futuro.

In [ ]:
LAGS = [1, 2, 4, 8, 12]
ROLLING_WINDOWS = [4, 8, 12]

feature_cols = [c for c in df_weekly.columns if c != 'nfci']  # nfci se trata aparte (autorregresivo)
all_predictor_cols = feature_cols + ['nfci']

df_features = df_weekly.copy()

for col in all_predictor_cols:
    for lag in LAGS:
        df_features[f'{col}_lag{lag}'] = df_weekly[col].shift(lag)
    shifted = df_weekly[col].shift(1)  # excluye la semana de referencia de las ventanas móviles
    for window in ROLLING_WINDOWS:
        df_features[f'{col}_roll{window}_mean'] = shifted.rolling(window).mean()
        df_features[f'{col}_roll{window}_std'] = shifted.rolling(window).std()

print(f'Dataset de features: {df_features.shape[0]} filas x {df_features.shape[1]} columnas')
df_features.tail()

In [ ]:
PROCESSED_DATA_DIR = 'data/processed'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

output_path = os.path.join(PROCESSED_DATA_DIR, 'ews_weekly_features.csv')
df_features.to_csv(output_path)
print(f'Dataset guardado en: {output_path}')

## 6. Modelización y evaluación

Se comparan tres modelos de complejidad creciente para predecir el NFCI `H` semanas vista:

1. **Ridge** — regresión lineal regularizada. Baseline interpretable: cada coeficiente tiene signo y magnitud directamente interpretables, y sirve de referencia mínima que cualquier modelo más complejo debe superar para justificar su uso.
2. **LightGBM** — gradient boosting sobre árboles. Captura no linealidades e interacciones entre variables sin necesidad de especificarlas a mano; es el modelo candidato principal por su buen equilibrio entre precisión, velocidad e interpretabilidad (SHAP, Sección 8).
3. **LSTM** — red neuronal recurrente. A diferencia de los dos anteriores, no recibe las variables rezagadas manualmente: se le pasa la secuencia semanal en bruto y es la propia red la que aprende qué dependencias temporales le son relevantes. Permite valorar si existe estructura temporal de largo plazo que los modelos anteriores, basados en features tabulares, no estén capturando.

**Validación**: al ser series temporales, ninguna validación puede barajar (`shuffle`) las observaciones. Se usa `TimeSeriesSplit` de scikit-learn (walk-forward, ventana expansiva) para ajustar hiperparámetros sobre el periodo de desarrollo, y un **holdout final** (los últimos ~2 años, nunca visto durante el ajuste) para la comparación definitiva entre modelos. El backtesting específico sobre GFC 2008, COVID 2020 y SVB 2023 se aborda por separado en la Sección 7, reentrenando cada modelo únicamente con la información disponible antes de cada episodio.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score
from lightgbm import LGBMRegressor
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

### 6.1 Horizonte de predicción y umbral de estrés

Se fija un horizonte principal de `H = 8` semanas (≈ 2 meses): suficiente antelación para que un equipo de riesgos actúe, y lo bastante corto para que la relación predictiva con las variables actuales siga siendo fuerte. La sensibilidad a otros horizontes (4 y 13 semanas) se explora en el backtesting (Sección 7).

El umbral de estrés se calcula con el **percentil 75 de una ventana expansiva** del NFCI: en cada semana `t`, el umbral vigente usa únicamente el historial de NFCI hasta `t` (nunca datos futuros), reproduciendo cómo funcionaría el sistema en producción (Sección 9). Se eligió P75 en vez de P90 porque, al incluir la GFC 2008 —con diferencia el episodio más extremo de la muestra— en el histórico de referencia, un umbral P90 queda calibrado casi exclusivamente por esos valores: ni COVID 2020 ni SVB 2023 llegan a cruzarlo nunca, por severos que fueran en su momento. P75 permite que el sistema también señale episodios de estrés moderado, no solo eventos de magnitud GFC.

In [ ]:
H = 8  # horizonte de predicción, en semanas
# P90 lo domina la GFC (con diferencia el episodio más extremo de la muestra): ni COVID ni SVB
# lo cruzan nunca, por lo que con P90 el recall es estructuralmente 0 fuera de la GFC. Se usa P75
# como umbral de "estrés elevado" para que el sistema también capture episodios moderados, no solo
# de magnitud GFC — más realista para un EWS que debe avisar pronto.
STRESS_PERCENTILE = 0.75
MIN_PERIODS_THRESHOLD = 104  # 2 años de historial mínimo antes de fijar un umbral

df_model = df_features.copy()
df_model['stress_threshold'] = df_weekly['nfci'].expanding(min_periods=MIN_PERIODS_THRESHOLD).quantile(STRESS_PERCENTILE)
df_model['target'] = df_weekly['nfci'].shift(-H)
df_model['target_stress'] = (df_model['target'] > df_model['stress_threshold']).astype(int)

# ted_spread (y sus lags/rolling) se excluye de las features de modelización: queda en NaN de forma
# permanente desde que la Fed discontinuó TEDRATE en 2022, y un dropna() que la incluyera borraría
# todas las semanas desde entonces (incluida por completo la ventana de SVB 2023). Se mantiene solo
# como referencia histórica en el EDA (Sección 4).
FEATURE_COLS = [c for c in df_features.columns if not c.startswith('ted_spread')]
model_df = df_model.dropna(subset=FEATURE_COLS + ['target', 'stress_threshold'])

X = model_df[FEATURE_COLS]
y_reg = model_df['target']
y_clf = model_df['target_stress']

print(f'Observaciones utilizables para modelizar: {len(model_df)}')
print(f'% semanas de estrés (umbral expansivo P{int(STRESS_PERCENTILE*100)}): {y_clf.mean()*100:.1f}%')

### 6.2 Split walk-forward y holdout final

In [ ]:
HOLDOUT_WEEKS = 104  # últimos ~2 años, reservados como test final

X_dev, X_holdout = X.iloc[:-HOLDOUT_WEEKS], X.iloc[-HOLDOUT_WEEKS:]
y_reg_dev, y_reg_holdout = y_reg.iloc[:-HOLDOUT_WEEKS], y_reg.iloc[-HOLDOUT_WEEKS:]
y_clf_dev, y_clf_holdout = y_clf.iloc[:-HOLDOUT_WEEKS], y_clf.iloc[-HOLDOUT_WEEKS:]
dates_holdout = X_holdout.index

tscv = TimeSeriesSplit(n_splits=5)

print(f'Desarrollo (CV walk-forward): {X_dev.index.min().date()} → {X_dev.index.max().date()}  ({len(X_dev)} semanas)')
print(f'Holdout (test final):        {X_holdout.index.min().date()} → {X_holdout.index.max().date()}  ({len(X_holdout)} semanas)')

### 6.3 Modelo baseline: Ridge

**Bondades**: rápido, estable, totalmente interpretable (coeficiente = efecto marginal de cada variable) y con muy bajo riesgo de sobreajuste gracias a la regularización L2.
**Debilidades**: asume relaciones lineales entre predictores y target, por lo que no captura interacciones ni umbrales no lineales — típicos en episodios de estrés financiero, donde el comportamiento del sistema cambia de régimen de forma abrupta.

In [ ]:
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge()),
])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid={'ridge__alpha': [0.1, 1, 5, 10, 50, 100, 200]},
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
ridge_grid.fit(X_dev, y_reg_dev)
ridge_model = ridge_grid.best_estimator_

print(f"Mejor alpha: {ridge_grid.best_params_['ridge__alpha']}")
print(f'RMSE walk-forward (CV desarrollo): {-ridge_grid.best_score_:.4f}')

### 6.4 LightGBM

**Bondades**: captura no linealidades e interacciones entre variables automáticamente, maneja bien variables con distinta escala, y ofrece herramientas de interpretabilidad robustas (SHAP, Sección 8). Suele ser el mejor compromiso precisión/interpretabilidad para datos tabulares de este tamaño.
**Debilidades**: más hiperparámetros que ajustar, mayor riesgo de sobreajuste con pocas observaciones (~1.300 semanas) si no se regulariza bien, y menos interpretable "de un vistazo" que Ridge.

In [ ]:
lgbm_grid = GridSearchCV(
    LGBMRegressor(random_state=42, verbosity=-1),
    param_grid={
        'n_estimators': [200, 400],
        'num_leaves': [15, 31],
        'learning_rate': [0.03, 0.05],
        'min_child_samples': [10, 20],
    },
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
lgbm_grid.fit(X_dev, y_reg_dev)
lgbm_model = lgbm_grid.best_estimator_

print(f'Mejores hiperparámetros: {lgbm_grid.best_params_}')
print(f'RMSE walk-forward (CV desarrollo): {-lgbm_grid.best_score_:.4f}')

Importancia de variables nativa de LightGBM (el análisis SHAP detallado por tipo de crisis se realiza en la Sección 8).

In [ ]:
importances = pd.Series(lgbm_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
importances.sort_values().plot(kind='barh', color='#4c72b0')
plt.title('LightGBM — Top 15 variables más relevantes')
plt.xlabel('Importancia (número de splits)')
plt.tight_layout()
plt.show()

### 6.5 LSTM

**Bondades**: modela dependencias temporales de forma end-to-end, sin necesidad de diseñar manualmente qué lags o ventanas móviles importan; puede capturar patrones de más largo plazo o no lineales en la evolución temporal que los lags fijos de Ridge/LightGBM no representan.
**Debilidades**: requiere más datos para generalizar bien (aquí el dataset es pequeño para un estándar de deep learning, ~1.300 semanas), es la menos interpretable de las tres, y su entrenamiento es más sensible a la inicialización y a los hiperparámetros.

A diferencia de Ridge y LightGBM, la LSTM **no** recibe los lags y estadísticos móviles construidos a mano en la Sección 5: se le pasa directamente la secuencia semanal de los niveles brutos (`df_weekly`), y es la propia recurrencia la que debe aprender la estructura temporal. Mezclar ambos enfoques sería redundante y no aporta señal adicional.

In [ ]:
SEQ_LEN = 12  # semanas de contexto que ve la LSTM en cada predicción

raw_cols = [c for c in df_weekly.columns if c != 'ted_spread']  # excluida: discontinuada en 2022 (ver Sección 6.1)
lstm_base = df_weekly.copy()
lstm_base['target'] = df_weekly['nfci'].shift(-H)
lstm_base = lstm_base.dropna(subset=raw_cols + ['target'])


def build_sequences(df, feature_cols, target_col, seq_len):
    values, targets = df[feature_cols].values, df[target_col].values
    X_seq = np.stack([values[i - seq_len:i] for i in range(seq_len, len(df))])
    y_seq = targets[seq_len:]
    idx = df.index[seq_len:]
    return X_seq, y_seq, idx


X_seq, y_seq, seq_dates = build_sequences(lstm_base, raw_cols, 'target', SEQ_LEN)

# La LSTM no usa lags/rolling manuales, así que su propio dropna puede conservar semanas puntuales
# que Ridge/LightGBM pierden (un hueco de datos en una serie se propaga varias semanas hacia
# delante en sus columnas de lags). Por eso las fechas de desarrollo/holdout de la LSTM se alinean
# explícitamente contra las de X_dev/dates_holdout (intersección), en lugar de asumir que ambos
# conjuntos son idénticos.
dev_dates = set(X_dev.index)
holdout_dates = set(dates_holdout)

train_mask = np.array([d in dev_dates for d in seq_dates])
holdout_mask = np.array([d in holdout_dates for d in seq_dates])

X_seq_dev, X_seq_holdout = X_seq[train_mask], X_seq[holdout_mask]
y_seq_dev, y_seq_holdout = y_seq[train_mask], y_seq[holdout_mask]
seq_dates_holdout = seq_dates[holdout_mask]

print(f'Semanas de holdout comunes a Ridge/LightGBM y LSTM: {len(seq_dates_holdout)} de {len(dates_holdout)}')

n_features = X_seq_dev.shape[2]
scaler_lstm = StandardScaler().fit(X_seq_dev.reshape(-1, n_features))


def scale_sequences(seq, scaler, n_feat):
    shape = seq.shape
    return scaler.transform(seq.reshape(-1, n_feat)).reshape(shape)


X_seq_dev_scaled = scale_sequences(X_seq_dev, scaler_lstm, n_features)
X_seq_holdout_scaled = scale_sequences(X_seq_holdout, scaler_lstm, n_features)

print(f'Secuencias de desarrollo: {X_seq_dev_scaled.shape}, holdout: {X_seq_holdout_scaled.shape}')

In [ ]:
val_split = int(len(X_seq_dev_scaled) * 0.85)  # último 15% del desarrollo como validación interna (early stopping)
X_seq_train, X_seq_val = X_seq_dev_scaled[:val_split], X_seq_dev_scaled[val_split:]
y_seq_train, y_seq_val = y_seq_dev[:val_split], y_seq_dev[val_split:]

lstm_model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, n_features)),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
])
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_seq_train, y_seq_train,
    validation_data=(X_seq_val, y_seq_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Entrenamiento detenido tras {len(history.history['loss'])} épocas (early stopping)")

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('LSTM — curva de aprendizaje (MSE)')
plt.xlabel('Época')
plt.legend()
plt.tight_layout()
plt.show()

### 6.6 Comparativa de modelos sobre el holdout final

In [ ]:
def evaluate_regression(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {'modelo': label, 'RMSE': rmse, 'MAE': mae}


def evaluate_classification(y_true_stress, y_pred_reg, threshold_series, label):
    y_pred_stress = (np.asarray(y_pred_reg) > np.asarray(threshold_series)).astype(int)
    return {
        'modelo': label,
        'Precision': precision_score(y_true_stress, y_pred_stress, zero_division=0),
        'Recall': recall_score(y_true_stress, y_pred_stress, zero_division=0),
        'F1': f1_score(y_true_stress, y_pred_stress, zero_division=0),
    }


ridge_pred = ridge_model.predict(X_holdout)
lgbm_pred = lgbm_model.predict(X_holdout)
lstm_pred = lstm_model.predict(X_seq_holdout_scaled, verbose=0).flatten()

threshold_holdout = model_df.loc[dates_holdout, 'stress_threshold']
# la LSTM puede tener alguna semana menos que Ridge/LightGBM (ver Sección 6.5), así que se evalúa
# sobre su propio conjunto de fechas (seq_dates_holdout), no sobre dates_holdout directamente
threshold_holdout_lstm = model_df.loc[seq_dates_holdout, 'stress_threshold']
y_clf_holdout_lstm = model_df.loc[seq_dates_holdout, 'target_stress']

reg_results = pd.DataFrame([
    evaluate_regression(y_reg_holdout, ridge_pred, 'Ridge'),
    evaluate_regression(y_reg_holdout, lgbm_pred, 'LightGBM'),
    evaluate_regression(y_seq_holdout, lstm_pred, 'LSTM'),
]).set_index('modelo')

clf_results = pd.DataFrame([
    evaluate_classification(y_clf_holdout, ridge_pred, threshold_holdout, 'Ridge'),
    evaluate_classification(y_clf_holdout, lgbm_pred, threshold_holdout, 'LightGBM'),
    evaluate_classification(y_clf_holdout_lstm, lstm_pred, threshold_holdout_lstm, 'LSTM'),
]).set_index('modelo')

print('Métricas de regresión (holdout):')
display(reg_results.round(4))
print('\nMétricas de clasificación de estrés (holdout):')
display(clf_results.round(4))

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(dates_holdout, y_reg_holdout, label='NFCI real', color='black', linewidth=1.5)
plt.plot(dates_holdout, ridge_pred, label='Ridge', alpha=0.8)
plt.plot(dates_holdout, lgbm_pred, label='LightGBM', alpha=0.8)
plt.plot(seq_dates_holdout, lstm_pred, label='LSTM', alpha=0.8)  # puede tener alguna semana menos, ver celda anterior
plt.axhline(0, color='grey', linewidth=0.6, linestyle='--')
plt.title(f'NFCI real vs. predicho a {H} semanas — holdout {dates_holdout.min().date()} → {dates_holdout.max().date()}')
plt.legend()
plt.tight_layout()
plt.show()

**Sobre las métricas de clasificación en 0 en este holdout**: no es un error de código. El holdout cubre los últimos ~2 años (aprox. 2023-2025), un periodo sin episodios de estrés severo comparable a los de 2008/2020/2023 — es decir, `y_clf_holdout` apenas (o nunca) vale 1, y ninguna métrica de clasificación basada en positivos reales puede ser informativa ahí (precisión y recall son 0/0 por definición). Esto es exactamente el motivo por el que la Sección 7 evalúa la clasificación de estrés **específicamente sobre los tres episodios históricos**, que sí contienen semanas de estrés real: un holdout cronológico genérico no es el test adecuado para un sistema diseñado para detectar eventos raros.

## 7. Backtesting sobre episodios históricos

Para cada episodio (GFC 2008, COVID 2020, SVB 2023) se reentrena cada modelo usando **únicamente** datos anteriores al inicio de la ventana de evaluación —sin volver a buscar hiperparámetros, se reutilizan los ya encontrados en la Sección 6—, y se puntúa sobre una ventana que empieza `BACKTEST_BUFFER_WEEKS` semanas antes del inicio oficial de la crisis (para poder medir con margen si la señal se adelanta) y termina al final del episodio.

El **lead time** se define como el número de semanas entre la primera alerta del modelo (primera semana en la que la predicción supera el umbral de estrés vigente) y la semana de máximo estrés real (pico de NFCI) dentro del episodio.

In [ ]:
BACKTEST_BUFFER_WEEKS = 26  # margen antes del inicio oficial de cada episodio, para medir el lead time

RIDGE_BEST_ALPHA = ridge_grid.best_params_['ridge__alpha']
LGBM_BEST_PARAMS = lgbm_grid.best_params_


def fit_ridge(X_train, y_train):
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=RIDGE_BEST_ALPHA))])
    pipe.fit(X_train, y_train)
    return pipe


def fit_lgbm(X_train, y_train):
    model = LGBMRegressor(random_state=42, verbosity=-1, **LGBM_BEST_PARAMS)
    model.fit(X_train, y_train)
    return model


def fit_lstm(X_seq_train, y_seq_train):
    n_feat = X_seq_train.shape[2]
    scaler = StandardScaler().fit(X_seq_train.reshape(-1, n_feat))
    X_train_scaled = scale_sequences(X_seq_train, scaler, n_feat)

    val_split = int(len(X_train_scaled) * 0.85)
    X_tr, X_val = X_train_scaled[:val_split], X_train_scaled[val_split:]
    y_tr, y_val = y_seq_train[:val_split], y_seq_train[val_split:]

    model = models.Sequential([
        layers.Input(shape=(SEQ_LEN, n_feat)),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=100, batch_size=32,
              callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
    return model, scaler

In [ ]:
def true_target(dates):
    return df_weekly['nfci'].shift(-H).loc[dates]


def threshold_at(dates):
    return df_model['stress_threshold'].loc[dates]


def true_stress(dates):
    return (true_target(dates) > threshold_at(dates)).astype(int)


def find_peak_date(crisis_start, crisis_end):
    window = df_weekly.loc[crisis_start:crisis_end, 'nfci']
    return window.idxmax()


def lead_time_weeks(alarm_dates, peak_date):
    prior = alarm_dates[alarm_dates <= peak_date]
    if len(prior) == 0:
        return np.nan
    return (peak_date - prior.min()).days / 7


backtest_rows = []
backtest_predictions = {}
backtest_models = {}  # se reutilizan en la Sección 8 (SHAP por episodio), sin reentrenar

for crisis_label, (crisis_start, crisis_end) in CRISIS_PERIODS.items():
    crisis_start_ts, crisis_end_ts = pd.Timestamp(crisis_start), pd.Timestamp(crisis_end)
    eval_start = crisis_start_ts - pd.Timedelta(weeks=BACKTEST_BUFFER_WEEKS)

    train_dates = model_df.index[model_df.index < eval_start]
    eval_dates = model_df.index[(model_df.index >= eval_start) & (model_df.index <= crisis_end_ts)]

    X_train, y_train = model_df.loc[train_dates, FEATURE_COLS], model_df.loc[train_dates, 'target']
    X_eval = model_df.loc[eval_dates, FEATURE_COLS]

    ridge_bt = fit_ridge(X_train, y_train)
    lgbm_bt = fit_lgbm(X_train, y_train)

    seq_train_mask = seq_dates < eval_start
    seq_eval_mask = (seq_dates >= eval_start) & (seq_dates <= crisis_end_ts)
    lstm_bt, lstm_scaler_bt = fit_lstm(X_seq[seq_train_mask], y_seq[seq_train_mask])
    X_seq_eval_scaled = scale_sequences(X_seq[seq_eval_mask], lstm_scaler_bt, X_seq.shape[2])
    seq_eval_dates = seq_dates[seq_eval_mask]

    preds = {
        'Ridge': pd.Series(ridge_bt.predict(X_eval), index=eval_dates),
        'LightGBM': pd.Series(lgbm_bt.predict(X_eval), index=eval_dates),
        'LSTM': pd.Series(lstm_bt.predict(X_seq_eval_scaled, verbose=0).flatten(), index=seq_eval_dates),
    }
    backtest_predictions[crisis_label] = preds
    backtest_models[crisis_label] = {
        'Ridge': ridge_bt, 'LightGBM': lgbm_bt, 'LSTM': (lstm_bt, lstm_scaler_bt),
        'X_eval': X_eval,  # features tabulares de la ventana de evaluación, para SHAP (Sección 8)
    }

    peak_date = find_peak_date(crisis_start, crisis_end)

    for model_name, pred_series in preds.items():
        common_dates = pred_series.index
        y_true = true_target(common_dates)
        y_true_stress = true_stress(common_dates)
        thr = threshold_at(common_dates)

        rmse = np.sqrt(mean_squared_error(y_true, pred_series))
        mae = mean_absolute_error(y_true, pred_series)

        pred_stress = (pred_series > thr).astype(int)
        precision = precision_score(y_true_stress, pred_stress, zero_division=0)
        recall = recall_score(y_true_stress, pred_stress, zero_division=0)
        f1 = f1_score(y_true_stress, pred_stress, zero_division=0)

        alarm_dates = common_dates[pred_stress.values.astype(bool)]
        lead_time = lead_time_weeks(alarm_dates, peak_date)

        backtest_rows.append({
            'Episodio': crisis_label, 'Modelo': model_name,
            'RMSE': rmse, 'MAE': mae,
            'Precision': precision, 'Recall': recall, 'F1': f1,
            'Lead time (semanas)': lead_time,
        })

backtest_summary = pd.DataFrame(backtest_rows).set_index(['Episodio', 'Modelo'])
backtest_summary.round(3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 13))

for ax, (crisis_label, (crisis_start, crisis_end)) in zip(axes, CRISIS_PERIODS.items()):
    eval_start = pd.Timestamp(crisis_start) - pd.Timedelta(weeks=BACKTEST_BUFFER_WEEKS)
    plot_end = pd.Timestamp(crisis_end) + pd.Timedelta(weeks=8)
    actual = df_weekly.loc[eval_start:plot_end, 'nfci']

    ax.plot(actual.index, actual.values, color='black', linewidth=1.5, label='NFCI real')

    for model_name, pred_series in backtest_predictions[crisis_label].items():
        ax.plot(pred_series.index, pred_series.values, alpha=0.8, label=f'{model_name} (a {H} sem.)')

    threshold_plot = df_model.loc[actual.index, 'stress_threshold']
    ax.plot(threshold_plot.index, threshold_plot.values, color='red', linestyle=':', linewidth=1, label='Umbral de estrés')

    ax.axvspan(pd.Timestamp(crisis_start), pd.Timestamp(crisis_end), color='orange', alpha=0.15)
    ax.set_title(crisis_label)
    ax.legend(fontsize=8, ncol=3, loc='upper left')

plt.tight_layout()
plt.show()

La tabla y los gráficos anteriores son la pieza central del informe: permiten comparar, episodio a episodio, si el sistema hubiera detectado con antelación una crisis sistémica (GFC 2008), un shock exógeno (COVID 2020) y una crisis sectorial (SVB 2023) — y con cuántas semanas de margen en cada caso.

**Nota metodológica a incorporar en el informe**: si LightGBM falla en detectar la GFC (Precision/Recall/F1 = 0) pese a ser el modelo con mejor RMSE medio, no es un error — es una limitación estructural conocida de los modelos basados en árboles: no pueden extrapolar más allá del rango de valores visto en entrenamiento. Al entrenarse únicamente con datos anteriores a 2007, el modelo nunca observó spreads de crédito o VIX de la magnitud que alcanzaron en la GFC, y como mucho predice el valor de la hoja más extrema que aprendió. Ridge (lineal) y LSTM (activaciones continuas) sí extrapolan, y por eso detectan parcialmente ese episodio. Es un punto de discusión de negocio importante: el modelo más preciso en promedio puede ser el peor precisamente en el evento más severo, que es el que más importa para un sistema de alerta temprana.

## 8. Interpretabilidad (SHAP)

Se usan valores SHAP (SHapley Additive exPlanations) sobre LightGBM para explicar, de forma aditiva y por observación, qué variables empujan la predicción del NFCI hacia arriba o hacia abajo. Es el método de interpretabilidad estándar para modelos de árboles: a diferencia de la importancia nativa de LightGBM (Sección 6.4, que solo cuenta splits), SHAP indica también la **dirección** del efecto de cada variable y permite comparar la contribución exacta en episodios concretos — justo lo que necesita un comité de riesgos para entender *por qué* el modelo está avisando.

Primero se calcula la explicación global sobre el modelo entrenado en la Sección 6 (holdout reciente); después se compara, episodio a episodio, qué variables dominan la señal en cada tipo de crisis (sistémica, exógena, sectorial) usando los modelos ya entrenados en el backtesting de la Sección 7.

In [ ]:
import shap

explainer_global = shap.TreeExplainer(lgbm_model)
shap_values_global = explainer_global.shap_values(X_holdout)

shap.summary_plot(shap_values_global, X_holdout, max_display=15, show=False)
plt.tight_layout()
plt.show()

### Comparativa por tipo de crisis

Se reutilizan los modelos LightGBM entrenados específicamente para cada episodio en la Sección 7 (sin reentrenar), y se calculan sus valores SHAP sobre la propia ventana de evaluación de cada crisis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (crisis_label, info) in zip(axes, backtest_models.items()):
    explainer = shap.TreeExplainer(info['LightGBM'])
    shap_values = explainer.shap_values(info['X_eval'])
    mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False).head(10)

    ax.barh(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1], color='#4c72b0')
    ax.set_title(crisis_label)
    ax.set_xlabel('|SHAP| medio')

plt.suptitle('Variables que más impulsan la señal de LightGBM, por tipo de crisis', y=1.03)
plt.tight_layout()
plt.show()

Como cabía esperar tras el resultado de la Sección 7, el panel de GFC 2008 debería mostrar magnitudes de SHAP mucho menores que COVID 2020 y SVB 2023: si LightGBM nunca emitió una alerta en la GFC, es porque sus predicciones apenas se movían — y sin movimiento en la predicción, no puede haber una atribución grande a ninguna variable. Este gráfico es la confirmación visual, variable a variable, de la limitación de extrapolación discutida en la Sección 7.

El comentario de qué variables concretas dominan en cada episodio (spread de crédito, VIX, curva de tipos...) se redacta en el informe una vez se dispone del resultado real de esta ejecución.

## Próximos pasos

Con los artefactos exportados, la siguiente sesión de trabajo aborda:

- Descargar `artifacts/lgbm_model.pkl` y `artifacts/config.json` desde Colab y colocarlos en `streamlit_app/`.
- Desplegar el dashboard (Streamlit Community Cloud recomendado, no requiere Python local).
- Redacción del informe Word (20 caras) con los resultados reales de esta ejecución.
- Grabación del vídeo de presentación (máx. 5 min, MP4, <50MB).

In [ ]:
import json
import joblib

ARTIFACTS_DIR = 'artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

joblib.dump(lgbm_model, os.path.join(ARTIFACTS_DIR, 'lgbm_model.pkl'))

config = {
    'START_DATE': START_DATE,  # el dashboard recalcula el umbral con el mismo histórico completo
    'H': H,
    'STRESS_PERCENTILE': STRESS_PERCENTILE,
    'MIN_PERIODS_THRESHOLD': MIN_PERIODS_THRESHOLD,
    'LAGS': LAGS,
    'ROLLING_WINDOWS': ROLLING_WINDOWS,
    'FEATURE_COLS': FEATURE_COLS,
    'SERIES': SERIES,
}
with open(os.path.join(ARTIFACTS_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('Artefactos guardados en artifacts/. Descárgalos desde el panel de archivos de Colab '
      '(carpeta artifacts/, botón derecho → Descargar) para usarlos en el dashboard de Streamlit.')

## Próximos pasos

Con SHAP completado, la siguiente sesión de trabajo aborda:

- **Sección 9 — Dashboard Streamlit** conectado a la API de FRED, con actualización semanal del estado del sistema.
- Redacción del informe Word (20 caras) con los resultados reales de esta ejecución.
- Grabación del vídeo de presentación (máx. 5 min, MP4, <50MB).